In [1]:
import os
import pandas as pd
import kagglehub

In [2]:
path = kagglehub.dataset_download("anshumish/crop-yield-data-with-soil-and-weather-dataset")
print("Dataset downloaded to:", path)


100%|██████████| 457k/457k [00:00<00:00, 99.6MB/s]

Extracting files...
Dataset downloaded to: /root/.cache/kagglehub/datasets/anshumish/crop-yield-data-with-soil-and-weather-dataset/versions/1


In [3]:
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            print(os.path.join(root, file))

/root/.cache/kagglehub/datasets/anshumish/crop-yield-data-with-soil-and-weather-dataset/versions/1/state_soil_data.csv
/root/.cache/kagglehub/datasets/anshumish/crop-yield-data-with-soil-and-weather-dataset/versions/1/crop_yield.csv
/root/.cache/kagglehub/datasets/anshumish/crop-yield-data-with-soil-and-weather-dataset/versions/1/state_weather_data_1997_2020.csv


In [4]:
soil_path = os.path.join(path, 'state_soil_data.csv')
weather_path = os.path.join(path, 'state_weather_data_1997_2020.csv')
crop_path = os.path.join(path, 'crop_yield.csv')

In [5]:
# 4. Load the raw tables
df_soil = pd.read_csv(soil_path)
df_weather = pd.read_csv(weather_path)
df_crop = pd.read_csv(crop_path)

print("Soil shape:", df_soil.shape)
print("Weather shape:", df_weather.shape)
print("Crop shape:", df_crop.shape)

print("SOIL columns:", df_soil.columns.tolist())
print("WEATHER columns:", df_weather.columns.tolist())
print("CROP columns:", df_crop.columns.tolist())

display(df_soil.head(2))
display(df_weather.head(2))
display(df_crop.head(2))

Soil shape: (30, 5)
Weather shape: (720, 5)
Crop shape: (19689, 9)
SOIL columns: ['state', 'N', 'P', 'K', 'pH']
WEATHER columns: ['state', 'year', 'avg_temp_c', 'total_rainfall_mm', 'avg_humidity_percent']
CROP columns: ['crop', 'year', 'season', 'state', 'area', 'production', 'fertilizer', 'pesticide', 'yield']


,state,N,P,K,pH
0,Andhra Pradesh,78,45,22,6.8
1,Arunachal Pradesh,55,15,35,5.5


,state,year,avg_temp_c,total_rainfall_mm,avg_humidity_percent
0,Andhra Pradesh,1997,28.21,1191.08,69.56
1,Andhra Pradesh,1998,28.21,1100.41,71.95


,crop,year,season,state,area,production,fertilizer,pesticide,yield
0,Arecanut,1997,Whole Year,Assam,73814.0,56708,7024878.38,22882.34,0.796087
1,Arhar/Tur,1997,Kharif,Assam,6637.0,4685,631643.29,2057.47,0.710435


In [6]:
# ---------- Step 1: Rename to a common standard ----------
# SOIL: no year column – we'll keep only 'State' and nutrients
df_soil = df_soil.rename(columns={'state': 'State'})  # N, P, K, pH remain same

# WEATHER
df_weather = df_weather.rename(columns={
    'state': 'State',
    'year': 'Year',
    'avg_temp_c': 'Temperature',
    'total_rainfall_mm': 'Rainfall',
    'avg_humidity_percent': 'Humidity'
})

# CROP
df_crop = df_crop.rename(columns={
    'crop': 'Crop',
    'year': 'Year',
    'state': 'State'
})

# ---------- Step 2: Ensure Year is integer where present ----------
df_weather['Year'] = pd.to_numeric(df_weather['Year'], errors='coerce').astype('Int64')
df_crop['Year'] = pd.to_numeric(df_crop['Year'], errors='coerce').astype('Int64')

# ---------- Step 3: Drop duplicates in the soil table (just in case) ----------
# Keep the first entry for each state; soil is assumed constant over time.
df_soil = df_soil.drop_duplicates(subset='State', keep='first')

# ---------- Step 4: First merge: crop + weather on State and Year ----------
merged_crop_weather = pd.merge(df_crop, df_weather, on=['State', 'Year'], how='inner')

# ---------- Step 5: Then merge with soil on State only ----------
final_df = pd.merge(merged_crop_weather, df_soil, on='State', how='left')

# ---------- Step 6: Clean – drop rows with missing essential values ----------
essential_cols = ['N', 'P', 'K', 'pH', 'Temperature', 'Humidity', 'Rainfall', 'Crop']
final_df.dropna(subset=essential_cols, inplace=True)

# ---------- Step 7: Optional – keep only relevant columns ----------
final_df = final_df[['State', 'Year', 'Crop', 'N', 'P', 'K', 'pH',
                     'Temperature', 'Humidity', 'Rainfall']]

# ---------- Step 8: Save ----------
final_df.to_csv('merged_crop_data.csv', index=False)
print("✅ Dataset ready! Shape:", final_df.shape)
print(final_df.head())

✅ Dataset ready! Shape: (19689, 10)
   State  Year          Crop   N   P   K   pH  Temperature  Humidity  Rainfall
0  Assam  1997      Arecanut  60  18  38  5.8        22.41     70.71   1468.92
1  Assam  1997     Arhar/Tur  60  18  38  5.8        22.41     70.71   1468.92
2  Assam  1997   Castor seed  60  18  38  5.8        22.41     70.71   1468.92
3  Assam  1997      Coconut   60  18  38  5.8        22.41     70.71   1468.92
4  Assam  1997  Cotton(lint)  60  18  38  5.8        22.41     70.71   1468.92


In [12]:
import pandas as pd
import numpy as np
import joblib
import time
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.multioutput import ClassifierChain
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import hamming_loss, f1_score
import xgboost as xgb
import lightgbm as lgb

# 1. Load & Preprocess
df = pd.read_csv('merged_crop_data.csv')
df['Crop'] = df['Crop'].str.strip()
df.drop_duplicates(inplace=True)

# Filter rare crops (noise reduction)
crop_counts = df['Crop'].value_counts()
df = df[df['Crop'].isin(crop_counts[crop_counts >= 15].index)]

# 2. Grouping & Feature Engineering
group_cols = ['State', 'Year']
feature_cols = ['N', 'P', 'K', 'pH', 'Temperature', 'Humidity', 'Rainfall']

grouped = df.groupby(group_cols)['Crop'].apply(set).reset_index()
features_df = df.groupby(group_cols)[feature_cols].mean().reset_index()
data = grouped.merge(features_df, on=group_cols)

# Encoders
state_encoder = LabelEncoder()
data['State'] = state_encoder.fit_transform(data['State'])

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(data['Crop'])
X = data[['State', 'Year'] + feature_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Define the 3 Models
# Random Forest
rf_base = RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42)

# XGBoost
xgb_base = xgb.XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.1,
                             objective='binary:logistic', random_state=42)

# LightGBM
lgb_base = lgb.LGBMClassifier(n_estimators=150, max_depth=6, learning_rate=0.1,
                              random_state=42, verbose=-1)

models = {
    'Random Forest': ClassifierChain(rf_base, order='random', random_state=42),
    'XGBoost': ClassifierChain(xgb_base, order='random', random_state=42),
    'LightGBM': ClassifierChain(lgb_base, order='random', random_state=42)
}

# 4. Train and Compare
results = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    y_pred_prob = model.predict_proba(X_test)

    # Using 0.2 threshold for realistic multi-label coverage
    y_pred = (y_pred_prob >= 0.20).astype(int)

    results[name] = {
        'Hamming Loss': hamming_loss(y_test, y_pred),
        'Micro F1': f1_score(y_test, y_pred, average='micro'),
        'Model': model
    }

# Display Results
for name, metrics in results.items():
    print(f"\n{name}:")
    print(f"  Hamming Loss: {metrics['Hamming Loss']:.4f}")
    print(f"  Micro F1:     {metrics['Micro F1']:.4f}")

# 5. Save the best performing model (example: Random Forest or XGBoost)
# Usually, XGBoost or RF performs best on this type of tabular data
best_model_name = max(results, key=lambda x: results[x]['Micro F1'])
print(f"\n🏆 Best performing model: {best_model_name}")

joblib.dump(results[best_model_name]['Model'], 'best_crop_model.pkl')
joblib.dump(state_encoder, 'state_encoder.pkl')
joblib.dump(mlb, 'crop_binarizer.pkl')

# 6. Recommendation Function (Top-K)
def recommend_crops_ensemble(state, year, N, P, K, pH, temp, hum, rain, top_k=5):
    model = joblib.load('best_crop_model.pkl')
    state_enc = joblib.load('state_encoder.pkl')
    mlb_loader = joblib.load('crop_binarizer.pkl')

    state_val = state_enc.transform([state])[0]
    features = np.array([[state_val, year, N, P, K, pH, temp, hum, rain]])

    probs = model.predict_proba(features)[0]
    crop_probs = sorted(zip(mlb_loader.classes_, probs), key=lambda x: x[1], reverse=True)

    return crop_probs[:top_k]

# Example Test
# final_recommendations = recommend_crops_ensemble('Punjab', 2026, 90, 42, 43, 6.5, 28, 70, 800)

Training Random Forest...
Training XGBoost...
Training LightGBM...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut


Random Forest:
  Hamming Loss: 0.0762
  Micro F1:     0.9229

XGBoost:
  Hamming Loss: 0.0746
  Micro F1:     0.9230

LightGBM:
  Hamming Loss: 0.0694
  Micro F1:     0.9281

🏆 Best performing model: LightGBM
